In [2]:
from pyspark.sql import SparkSession
import getpass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_RDD") \
    .getOrCreate()

# Lấy SparkContext từ SparkSession để dùng RDD
sc = spark.sparkContext

print("Spark version:", sc.version)


Spark version: 4.0.1


# **Bài 1: Tính Điểm Đánh Giá Trung Bình và Tổng Số Lượt Đánh Giá Cho Mỗi Phim**

In [6]:
movies_rdd = sc.textFile("data/movies.txt")
ratings_rdd = sc.textFile("data/ratings_*.txt")
user_rdd = sc.textFile("data/users.txt")
occupation_rdd = sc.textFile("data/occupations.txt")

In [4]:
movies_rdd.take(5)

['1001,The Godfather (1972),Crime|Drama',
 '1002,The Shawshank Redemption (1994),Drama',
 "1003,Schindler's List (1993),Biography|Drama|History",
 '1004,Raging Bull (1980),Biography|Drama|Sport',
 '1005,Casablanca (1942),Drama|Romance|War']

In [7]:
#Mapper cho ratings
ratings_mapped = ratings_rdd.map(lambda x: (x.split(",")[1], (float(x.split(",")[2]),1)))
ratings_mapped.take(5)

[('1020', (4.5, 1)),
 ('1015', (3.5, 1)),
 ('1030', (4.0, 1)),
 ('1047', (3.0, 1)),
 ('1012', (4.5, 1))]

In [8]:
#Reduce tính tổng ratings và tổng lượt số ratings
sum_ratings = ratings_mapped.reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1]))
sum_ratings.take(5)

[('1020', (66.0, 18)),
 ('1015', (30.5, 7)),
 ('1037', (70.0, 18)),
 ('1040', (65.0, 18)),
 ('1025', (73.0, 18))]

In [9]:
#Tính avg ratings
avg_ratings = sum_ratings.mapValues(lambda x : (x[0]/x[1], x[1]))
avg_ratings.take(5)

[('1020', (3.6666666666666665, 18)),
 ('1015', (4.357142857142857, 7)),
 ('1037', (3.888888888888889, 18)),
 ('1040', (3.611111111111111, 18)),
 ('1025', (4.055555555555555, 18))]

In [10]:
#Mapper lấy movieId và Title
movies_mapped = movies_rdd.map(lambda x: (x.split(",")[0], x.split(",")[1]))
#Join movie
joined_data = movies_mapped.join(avg_ratings)
joined_data.take(5)

[('1013', ('The Godfather: Part II (1974)', (4.0, 17))),
 ('1025', ('The Terminator (1984)', (4.055555555555555, 18))),
 ('1039',
  ('The Lord of the Rings: The Return of the King (2003)',
   (3.8181818181818183, 11))),
 ('1043', ('No Country for Old Men (2007)', (3.888888888888889, 18))),
 ('1012', ('Psycho (1960)', (4.0, 2)))]

In [12]:
#Format result
def format_result(record):
  data = record[1]
  title = data[0]
  avg_rating = data[1][0]
  total_ratings = data[1][1]
  return f"{title}  AverageRating: {avg_rating:.2f}  (TotalRating:{total_ratings})"

result = joined_data.map(format_result)
result.collect()

['The Godfather: Part II (1974)  AverageRating: 4.00  (TotalRating:17)',
 'The Terminator (1984)  AverageRating: 4.06  (TotalRating:18)',
 'The Lord of the Rings: The Return of the King (2003)  AverageRating: 3.82  (TotalRating:11)',
 'No Country for Old Men (2007)  AverageRating: 3.89  (TotalRating:18)',
 'Psycho (1960)  AverageRating: 4.00  (TotalRating:2)',
 'Sunset Boulevard (1950)  AverageRating: 4.36  (TotalRating:7)',
 'E.T. the Extra-Terrestrial (1982)  AverageRating: 3.67  (TotalRating:18)',
 'The Lord of the Rings: The Fellowship of the Ring (2001)  AverageRating: 3.89  (TotalRating:18)',
 'Gladiator (2000)  AverageRating: 3.61  (TotalRating:18)',
 'Lawrence of Arabia (1962)  AverageRating: 3.44  (TotalRating:18)',
 'Fight Club (1999)  AverageRating: 3.50  (TotalRating:7)',
 'The Silence of the Lambs (1991)  AverageRating: 3.14  (TotalRating:7)',
 'The Social Network (2010)  AverageRating: 3.86  (TotalRating:7)',
 'Mad Max: Fury Road (2015)  AverageRating: 3.47  (TotalRating: